# Simple Convolutional 2D Layer

The **Convolutional layer** is a fundamental component widely used in **Computer Vision** tasks. It applies a filter (kernel) that slides across the input to extract spatial features such as edges, textures, or patterns.

---

# Parameters

## `input_matrix`

A 2D NumPy array representing the input data, such as an image. Each element in this array corresponds to a pixel or feature value in the input space.

The dimensions of the input matrix are typically represented as:

$$
\text{height} \times \text{width}
$$

---

## `kernel`

Another 2D NumPy array representing the **convolutional filter**.

The kernel is smaller than the input matrix and slides over it to perform the convolution operation. Each element in the kernel acts as a weight that modifies the input during convolution.

Kernel size is denoted as:

$$
\text{kernel\_height} \times \text{kernel\_width}
$$

---

## `padding`

An integer specifying the number of rows and columns of **zeros added around the input matrix**.

Padding controls the spatial dimensions of the output by:

- Allowing the kernel to process **edge elements**
- Optionally **preserving the original input size**

---

## `stride`

An integer representing the number of steps the kernel moves across the input matrix during convolution.

- **Stride = 1** → kernel moves one cell at a time  
- **Stride > 1** → kernel skips elements, reducing output size

---

# Implementation

## 1. Padding the Input

The input matrix is padded with zeros according to the specified padding value.

This increases the input dimensions and allows the kernel to cover border and corner elements.

---

## 2. Calculating Output Dimensions

The height and width of the output matrix are calculated as:

$$
\text{output\_height} =
\frac{\text{input\_height}_{padded} - \text{kernel\_height}}{\text{stride}} + 1
$$

$$
\text{output\_width} =
\frac{\text{input\_width}_{padded} - \text{kernel\_width}}{\text{stride}} + 1
$$

---

## 3. Performing Convolution

The convolution process follows these steps:

1. A **nested loop** iterates over all positions where the kernel can be applied to the padded input.
2. At each position, a **region of the input matrix** with the same size as the kernel is selected.
3. Perform **element-wise multiplication** between the kernel and the selected input region.
4. **Sum the results** to produce a single scalar value.
5. Store this value in the corresponding position of the output matrix.

Mathematically:

$$
y_{i,j} = \sum_{m}\sum_{n}
x_{i+m, j+n} \cdot k_{m,n}
$$

Where:

- $x$ is the input matrix  
- $k$ is the kernel  
- $y$ is the output matrix  

---

## 4. Output

The function returns the **output matrix**, which contains the results of applying the convolution operation across the entire input matrix.

Each value in the output represents the response of the kernel at a specific spatial location.

---

# Summary

A **2D convolution layer** performs three main operations:

1. **Padding** the input (optional)
2. **Sliding the kernel** across the input using the specified stride
3. **Computing weighted sums** at each position

This process allows convolutional networks to **extract meaningful spatial features** from images and other grid-like data.

[Problem Desc](https://www.deep-ml.com/problems/41?from=Deep%20Learning)

In [1]:
import numpy as np

def simple_conv2d(input_matrix: np.ndarray, kernel: np.ndarray, padding: int, stride: int):
    input_height, input_width = input_matrix.shape
    kernel_height, kernel_width = kernel.shape
    
    input_matrix_padded = np.pad(input_matrix, pad_width=padding)
    
    input_height_padded, input_width_padded = input_matrix_padded.shape
    
    output_height = ((input_height_padded - kernel_height) // stride) + 1
    output_width = ((input_width_padded - kernel_width) // stride) + 1
    
    output_matrix = np.zeros((output_height, output_width))
    
    for i in range(output_height):
        for j in range(output_width):
            row_start = i * stride
            row_end = row_start + kernel_height
            col_start = j * stride
            col_end = col_start + kernel_width
            
            region = input_matrix_padded[row_start:row_end, col_start:col_end]
            
            output_matrix[i, j] = np.sum(region * kernel)
    
    return output_matrix

In [2]:
# Test Case 1
input_matrix = np.array([
    [1., 2., 3., 4., 5.],
    [6., 7., 8., 9., 10.],
    [11., 12., 13., 14., 15.],
    [16., 17., 18., 19., 20.],
    [21., 22., 23., 24., 25.],
])
kernel = np.array([
    [1., 2.],
    [3., -1.],
])
padding, stride = 0, 1
expected = np.array([
    [ 16., 21., 26., 31.],
    [ 41., 46., 51., 56.],
    [ 66., 71., 76., 81.],
    [ 91., 96., 101., 106.],
])
output = simple_conv2d(input_matrix, kernel, padding, stride)
print(output)

# Expected Output:
# [[ 16.,  21.,  26.,  31.],
# [ 41.,  46.,  51.,  56.],
# [ 66.,  71.,  76.,  81.],
# [ 91.,  96., 101., 106.]]

[[ 16.  21.  26.  31.]
 [ 41.  46.  51.  56.]
 [ 66.  71.  76.  81.]
 [ 91.  96. 101. 106.]]


In [3]:
# Test Case 2
input_matrix = np.array([
    [1., 2., 3., 4., 5.],
    [6., 7., 8., 9., 10.],
    [11., 12., 13., 14., 15.],
    [16., 17., 18., 19., 20.],
    [21., 22., 23., 24., 25.],
])
kernel = np.array([
    [.5, 3.2],
    [1., -1.],
])
padding, stride = 2, 2
expected = np.array([
        [ -1., 1., 3., 5., 7., 15.],
        [ -4., 16., 21., 26., 31., 35.],
        [  1., 41., 46., 51., 56., 55.],
        [  6., 66., 71., 76., 81., 75.],
        [ 11., 91., 96., 101., 106., 95.],
        [ 42., 65., 68., 71., 74.,  25.],
    ])
output = simple_conv2d(input_matrix, kernel, padding, stride)
print(output)

# Expected Output:
# [[ 0.,   0.,   0.,   0. ],
# [ 0.,   5.9, 13.3, 12.5],
# [ 0.,  42.9, 50.3, 27.5],
# [ 0.,  80.9, 88.3, 12.5],]

[[ 0.   0.   0.   0. ]
 [ 0.   5.9 13.3 12.5]
 [ 0.  42.9 50.3 27.5]
 [ 0.  80.9 88.3 12.5]]


# PyTorch Implementation

In [4]:
import torch
import torch.nn.functional as F

def simple_conv2d_torch(input_matrix: torch.Tensor, kernel: torch.Tensor, padding: int, stride: int) -> torch.Tensor:
    """
    Perform a 2D convolution on a single-channel input using PyTorch's built-in conv2d.
    input_matrix: 2D tensor (H, W)
    kernel: 2D tensor (kH, kW)
    padding: int, zero-padding on all sides
    stride: int, stride of the convolution
    """
    # Hint: conv2d expects input of shape (N, C, H, W) and weight of shape (out_channels, in_channels, kH, kW)
    h, w = input_matrix.shape
    kh, kw = kernel.shape
    
    input_4d = input_matrix.view(1, 1, h, w)
    kernel_4d = kernel.view(1, 1, kh, kw)
    
    output_4d = F.conv2d(input_4d, kernel_4d, padding=padding, stride=stride)
    
    output_2d = output_4d.squeeze()
    
    return output_2d

In [5]:
# Test Case 1
import torch
from __main__ import simple_conv2d
input_matrix = torch.tensor([
    [1., 2., 3., 4., 5.],
    [6., 7., 8., 9., 10.],
    [11., 12., 13., 14., 15.],
    [16., 17., 18., 19., 20.],
    [21., 22., 23., 24., 25.],
])
kernel = torch.tensor([
    [1., 2.],
    [3., -1.],
])
output = simple_conv2d_torch(input_matrix, kernel, padding=0, stride=1)
print(output)

# Expected Output:
# tensor([[ 16.,  21.,  26.,  31.],
#        [ 41.,  46.,  51.,  56.],
#        [ 66.,  71.,  76.,  81.],
 #       [ 91.,  96., 101., 106.]])

tensor([[ 16.,  21.,  26.,  31.],
        [ 41.,  46.,  51.,  56.],
        [ 66.,  71.,  76.,  81.],
        [ 91.,  96., 101., 106.]])


In [6]:
# Test Case 2
import torch
from __main__ import simple_conv2d
input_matrix = torch.tensor([
    [1., 2., 3., 4., 5.],
    [6., 7., 8., 9., 10.],
    [11., 12., 13., 14., 15.],
    [16., 17., 18., 19., 20.],
    [21., 22., 23., 24., 25.],
])
kernel = torch.tensor([
    [0.5, 3.2],
    [1.0, -1.0],
])
output = simple_conv2d_torch(input_matrix, kernel, padding=2, stride=2)
print(output)

# Expected Output:
# tensor([[ 0.0000,  0.0000,  0.0000,  0.0000],
#        [ 0.0000,  5.9000, 13.3000, 12.5000],
#        [ 0.0000, 42.9000, 50.3000, 27.5000],
#        [ 0.0000, 80.9000, 88.3000, 12.5000]])

tensor([[ 0.0000,  0.0000,  0.0000,  0.0000],
        [ 0.0000,  5.9000, 13.3000, 12.5000],
        [ 0.0000, 42.9000, 50.3000, 27.5000],
        [ 0.0000, 80.9000, 88.3000, 12.5000]])
